# GraphKV 第三阶段：真实 8B 的计算对照与位置消融

目标：① v5 的分块缓存，是否与一次完整前向的显式图注意力得到相近结果？② 影响问答的是连边，还是共享位置？③ BF16/后端/排列造成多大差异？

此 Notebook **尚未在你的 GPU 上执行**。不预填结果，不修改 v5。沿用本地模型、物理 GPU 3、BF16。重启内核，释放前两个 Notebook 的模型后从头运行。两种后端顺序加载，每次只保留一个模型。

默认运行 eager 和 SDPA。全部生成可能需要数分钟；每个条件打印进度并逐步写 JSON，方便保留中间结果。显式 mask 只适用于短例子，不用于完整论文的大规模评测。

参考：[论文 v2](https://arxiv.org/html/2506.07334v2#S3.SS1)、[已核对 v5 commit](https://github.com/zmsnhcy/GraphKV_refactor_v5/tree/1af09232d361d8f529d951f35348664aad37d415/GraphKV_refactor_v5)。


## 1. 环境与配置
位置编号由图实验统一控制。不要同时运行其他占用该卡的模型。


In [1]:
import os, sys
from pathlib import Path
GPU_ID = "3"
REPO_DIR = Path("/home/lyh/hcy_work/GraphKV_refactor_v5")  # 改为含两个 graph_kv 模块的目录
MODEL_PATH = Path("/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT")
BACKEND = "eager"  # 默认不依赖 flash-attn；也可用 eager / flash_attention_2
MAX_NEW_TOKENS = 80
if "torch" in sys.modules and sys.modules["torch"].cuda.is_initialized():
    raise RuntimeError("CUDA 已初始化，请重启内核，再从本格运行。")
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# GitHub 仓库外层可能还有一层同名目录。
if not (REPO_DIR / "graph_kv_cache.py").is_file():
    nested = REPO_DIR / "GraphKV_refactor_v5"
    if (nested / "graph_kv_cache.py").is_file():
        REPO_DIR = nested
assert (REPO_DIR / "graph_kv_cache.py").is_file(), f"请修改 REPO_DIR: {REPO_DIR}"
assert (MODEL_PATH / "config.json").is_file(), f"找不到本地模型: {MODEL_PATH}"
sys.path.insert(0, str(REPO_DIR.resolve()))
print("模块目录:", REPO_DIR, "模型目录:", MODEL_PATH, "物理 GPU:", GPU_ID, sep="\n")


模块目录:
/home/lyh/hcy_work/GraphKV_refactor_v5
模型目录:
/home/lyh/hcy_work/GraphKV/models/Tulu3-Block-FT
物理 GPU:
3


In [2]:
import json, inspect, gc, time
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from graph_kv_cache import GraphKVCache
from graph_kv_adapter import greedy_generate_graphkv, graphkv_next_logits, build_graphkv_from_ids
from IPython.display import display, HTML
assert transformers.__version__ == "4.50.0", "v5 adapter 要求 transformers==4.50.0，请选择原有兼容内核。"
assert torch.cuda.is_available(), "当前内核没有可用 CUDA"
assert torch.cuda.device_count() == 1, "单卡屏蔽未生效，请重启内核"
assert torch.cuda.is_bf16_supported(), "此测试默认使用 BF16"
print("torch:", torch.__version__, "transformers:", transformers.__version__)
print("实际导入:", inspect.getfile(GraphKVCache))
print("GPU:", torch.cuda.get_device_name(0))
def memory(label):
    free, total = torch.cuda.mem_get_info(0)
    print(f"{label}: free={free/2**30:.2f}/{total/2**30:.2f} GiB; "
          f"allocated={torch.cuda.memory_allocated()/2**30:.2f} GiB; "
          f"peak={torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
memory("加载前")


/home/lyh/miniconda3/envs/graphkv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.5.1+cu121 transformers: 4.50.0
实际导入: /home/lyh/hcy_work/GraphKV_refactor_v5/graph_kv_cache.py
GPU: NVIDIA GeForce RTX 4090
加载前: free=23.26/23.64 GiB; allocated=0.00 GiB; peak=0.00 GiB


In [3]:
SOURCE_IDS = ["4", "5"]
TARGET_IDS = ["1", "2", "3"]
EDGES = [("4", "1"), ("4", "3"), ("5", "2"), ("5", "3")]
source_texts = {
    "4": "[Document 4] In this fictional dataset, ALPHA supplies copper. Its code is AX17.\n",
    "5": "[Document 5] In this fictional dataset, BETA supplies silver. Its code is BY29.\n",
}
target_texts = {
    "1": "[Document 1] Project ORION uses the material supplied by ALPHA.\n",
    "2": "[Document 2] Project LYRA uses the material supplied by BETA.\n",
    "3": "[Document 3] Project NOVA combines the materials supplied by ALPHA and BETA.\n",
}
PREFIX = "<|user|>\nAnswer using the following documents. Give a short factual answer.\n"
QUERY = "\nQuestion: What material does ORION use, what material does LYRA use, and which two materials does NOVA combine?\n<|assistant|>\n"
assert set(source_texts).isdisjoint(target_texts)
assert all(s in source_texts and t in target_texts for s,t in EDGES)
print("边:", EDGES)
for n, text in {**source_texts, **target_texts}.items():
    print(n, "source" if n in SOURCE_IDS else "target", text)
print("target <- sources:", {t:[s for s,d in EDGES if d==t] for t in TARGET_IDS})
display(HTML("<pre>4 ─────► 1\n│\n└──────► 3 ◄──────┐\n                   │\n5 ─────► 2         │\n└──────────────────┘</pre>"))


边: [('4', '1'), ('4', '3'), ('5', '2'), ('5', '3')]
4 source [Document 4] In this fictional dataset, ALPHA supplies copper. Its code is AX17.

5 source [Document 5] In this fictional dataset, BETA supplies silver. Its code is BY29.

1 target [Document 1] Project ORION uses the material supplied by ALPHA.

2 target [Document 2] Project LYRA uses the material supplied by BETA.

3 target [Document 3] Project NOVA combines the materials supplied by ALPHA and BETA.

target <- sources: {'1': ['4'], '2': ['5'], '3': ['4', '5']}


In [4]:
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), local_files_only=True)
config = AutoConfig.from_pretrained(str(MODEL_PATH), local_files_only=True)
assert config.model_type == "llama"
rope = config.rope_scaling or {}
assert rope.get("rope_type", rope.get("type", "default")) in ("default", "llama3", "linear")
def tokenize(text):
    return tokenizer(text, return_tensors="pt", add_special_tokens=False,
                     truncation=False).input_ids
node_inputs = {n:tokenize(t) for n,t in {**source_texts, **target_texts}.items()}
prefix_ids, query_ids = tokenize(PREFIX), tokenize(QUERY)
lengths = {n:int(x.shape[1]) for n,x in node_inputs.items()}
P = int(prefix_ids.shape[1])
L = max(lengths.values())
Q = int(query_ids.shape[1])
assert P + 2*L + Q + MAX_NEW_TOKENS <= config.max_position_embeddings
print("P, L, Q =", P, L, Q)
print("node | kind | tokens | logical range [start,end)")
for n in SOURCE_IDS + TARGET_IDS:
    start = P if n in SOURCE_IDS else P+L
    print(n, "source" if n in SOURCE_IDS else "target", lengths[n], (start,start+lengths[n]))
print("query logical:", (P+2*L, P+2*L+Q))
print("query physical start:", P+sum(lengths.values()))
print("Tulu markers:", {x:tokenizer.encode(x, add_special_tokens=False) for x in ["<|user|>","<|assistant|>"]})


P, L, Q = 17 21 31
node | kind | tokens | logical range [start,end)
4 source 20 (17, 37)
5 source 21 (17, 38)
1 target 15 (38, 53)
2 target 16 (38, 54)
3 target 18 (38, 56)
query logical: (59, 90)
query physical start: 107
Tulu markers: {'<|user|>': [27, 91, 882, 91, 29], '<|assistant|>': [27, 91, 78191, 91, 29]}


In [5]:
from transformers import AutoModelForCausalLM
from graph_kv_adapter import build_graphkv_from_ids
import re, hashlib
BACKENDS = ["eager", "sdpa"]
RUN_GENERATION = True
MAX_NEW_TOKENS = 80
OUTPUT_DIR = Path.cwd()/"graphkv_stage3_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
QUESTIONS = {
 "ORION":"What material does ORION use? Answer with the material name only.",
 "LYRA":"What material does LYRA use? Answer with the material name only.",
 "NOVA":"Which two materials does NOVA combine? Answer with the material names only.",
 "COMBINED":"What material does ORION use, what material does LYRA use, and which two materials does NOVA combine?",
}
# 固定原 legacy 格式，避免模板换行与核心变量混在一起。
def query_text(question):
    return "\nQuestion: "+question+"\n<|assistant|>\n"
CONDITIONS = {
 "graph_shared":("graph","shared"),
 "noedges_shared":("independent","shared"),
 "graph_continuous":("graph","continuous"),
 "noedges_continuous":("independent","continuous"),
 "ordinary_sequence":("causal","continuous"),
}
results={"settings":{"model_path":str(MODEL_PATH),"repo_dir":str(REPO_DIR),
 "physical_gpu":GPU_ID,"dtype":"bfloat16","backends":BACKENDS,
 "torch":torch.__version__,"transformers":transformers.__version__,
 "max_new_tokens":MAX_NEW_TOKENS,"prefix":PREFIX,"source_texts":source_texts,
 "target_texts":target_texts,"edges":EDGES},
 "oracle":[],"stability":[],"answers":[],"candidate_scores":[]}
def save_results():
    path=OUTPUT_DIR/"graphkv_stage3_results.json"
    tmp=path.with_suffix(".tmp")
    tmp.write_text(json.dumps(results,ensure_ascii=False,indent=2),encoding="utf-8")
    tmp.replace(path)
print("实验条件:",CONDITIONS)
print("生成次数:",len(BACKENDS)*(len(CONDITIONS)+2)*len(QUESTIONS) if RUN_GENERATION else 0)


实验条件: {'graph_shared': ('graph', 'shared'), 'noedges_shared': ('independent', 'shared'), 'graph_continuous': ('graph', 'continuous'), 'noedges_continuous': ('independent', 'continuous'), 'ordinary_sequence': ('causal', 'continuous')}
生成次数: 56


## 2. 独立的完整图计算
这里不调用 GraphKVCache 的拼接或切片功能。先把所有 token 放在一个张量中，手写“谁可以读谁”的 mask。每个节点内部只能读取自己之前的 token；有边时 target 可以读取指定 source；query 读取 prefix 和全部节点。

共享位置：source 从 P 开始，target 从 P+L 开始，query 从 P+2L 开始。连续位置：每个 token 使用其物理顺序编号。两者的文本、节点顺序、mask 保持一致。

普通顺序基线另外使用全部因果可见关系，因此 prefix 和前面的文档也可被后续文档读取。它与图实验有多处差别，不能单独用于解释“共享位置”的影响。


In [6]:
def make_layout(query_ids, attention="graph", positions="shared"):
    chunks=[("prefix",prefix_ids)] + [(n,node_inputs[n]) for n in SOURCE_IDS+TARGET_IDS] + [("query",query_ids)]
    ranges={}; offsets=0; pos=[]; tensors=[]
    for name,ids in chunks:
        length=int(ids.shape[1]); ranges[name]=(offsets,offsets+length)
        if positions=="continuous":start=offsets
        elif name=="prefix":start=0
        elif name in SOURCE_IDS:start=P
        elif name in TARGET_IDS:start=P+L
        else:start=P+2*L
        pos.extend(range(start,start+length)); offsets+=length; tensors.append(ids.cpu())
    allow=torch.zeros((offsets,offsets),dtype=torch.bool)
    if attention=="causal":
        allow=torch.ones_like(allow).tril()
    else:
        for name,ids in chunks:
            a,b=ranges[name]; allow[a:b,a:b]=torch.ones((b-a,b-a),dtype=torch.bool).tril()
            if name=="query":allow[a:b,:a]=True
            elif attention=="graph" and name in TARGET_IDS:
                for s,t in EDGES:
                    if t==name:
                        c,d=ranges[s];allow[a:b,c:d]=True
    return torch.cat(tensors,-1),torch.tensor([pos]),allow,ranges
@torch.inference_mode()
def dense_forward(model,query_ids,attention="graph",positions="shared",keep_cache=False):
    ids,pos,allow,ranges=make_layout(query_ids,attention,positions)
    device=model.get_input_embeddings().weight.device
    mask=torch.zeros((1,1,ids.shape[1],ids.shape[1]),dtype=next(model.parameters()).dtype,device=device)
    mask.masked_fill_(~allow.to(device)[None,None],torch.finfo(mask.dtype).min)
    out=model(ids.to(device),position_ids=pos.to(device),
        cache_position=torch.arange(ids.shape[1],device=device),attention_mask=mask,
        use_cache=keep_cache,return_dict=True)
    return out,ranges
def check_mask_contract():
    q=tokenize(query_text(QUESTIONS["ORION"]))
    _,pos,a,r=make_layout(q)
    for target in TARGET_IDS:
        x,y=r[target]
        for source in SOURCE_IDS:
            u,v=r[source]
            assert bool(a[x:y,u:v].all())==((source,target) in EDGES)
        u,v=r["prefix"];assert not bool(a[x:y,u:v].any())
    x,y=r["query"];assert bool(a[x:y,:x].all())
    assert pos[0,x].item()==P+2*L
    _,pc,ac,rc=make_layout(q,"graph","continuous")
    assert torch.equal(a,ac) and r==rc
    assert torch.equal(pc,torch.arange(pc.shape[1]).unsqueeze(0))
    print("PASS: 4→1、5→2、4/5→3、prefix 隔离、query 可见与位置控制")
check_mask_contract()


PASS: 4→1、5→2、4/5→3、prefix 隔离、query 可见与位置控制


## 3. 比较方法与停止原因
不套用小模型 FP32 的严格误差阈值到 8B BF16。保存每层 K/V 的最大与平均差、logits 差、概率分布差和前几个候选。

同顺序重复计算提供误差参照。排列不变测试比较的是 v5 已编码节点的拼接顺序，不改变节点本身。首 token 相同也不等于完整答案相同。

自由回答的材料检查只用于提示；合并问题必须人工检查项目对应关系。


In [7]:
def difference(a,b):
    d=(a.detach().float()-b.detach().float()).abs()
    return {"max_abs":d.max().item(),"mean_abs":d.mean().item()}
def compare_logits(a,b):
    a=a.detach().float();b=b.detach().float()
    pa=a.softmax(-1);pb=b.softmax(-1)
    la=a.log_softmax(-1);lb=b.log_softmax(-1)
    row=difference(a,b)
    row.update({"KL_a_b":(pa*(la-lb)).sum(-1).mean().item(),
        "prob_L1":(pa-pb).abs().sum(-1).mean().item(),
        "argmax_a":a.argmax(-1).item(),"argmax_b":b.argmax(-1).item(),
        "top5_a":a[0].topk(5).indices.tolist(),"top5_b":b[0].topk(5).indices.tolist()})
    return row
def cache_hash(g):
    h=hashlib.sha256()
    for n in SOURCE_IDS+TARGET_IDS:
        for attr in ["key_cache","value_cache"]:
            for x in getattr(g.nodes[n],attr):
                h.update(x.detach().cpu().contiguous().view(torch.uint8).numpy().tobytes())
    return h.hexdigest()
def eos_ids(model):
    ids=model.generation_config.eos_token_id
    if ids is None:ids=tokenizer.eos_token_id
    return set([] if ids is None else ([ids] if isinstance(ids,int) else ids))
@torch.inference_mode()
def dense_generate(model,question,attention,positions):
    ids=tokenize(query_text(question)); tokens=[]; stopped=False
    eos=eos_ids(model)
    for _ in range(MAX_NEW_TOKENS):
        out,_=dense_forward(model,ids,attention,positions)
        token=out.logits[0,-1].argmax().item();del out
        tokens.append(token)
        if token in eos:stopped=True;break
        ids=torch.cat([ids,torch.tensor([[token]],dtype=torch.long)],-1)
    return {"answer":tokenizer.decode(tokens,skip_special_tokens=True),"token_ids":tokens,
        "eos":stopped,"hit_limit":len(tokens)==MAX_NEW_TOKENS and not stopped}
def rough_check(project,text):
    if project=="COMBINED":return None
    found={w for w in ["copper","silver"] if re.search(r"\b"+w+r"\b",text.lower())}
    expected={"ORION":{"copper"},"LYRA":{"silver"},"NOVA":{"copper","silver"}}
    return found==expected[project]
@torch.inference_mode()
def candidate_score(model,question,candidate,attention,positions):
    # 使用与自由生成相同的问题；包括大小写和空格变体，不将它们的概率直接相加。
    q=tokenize(query_text(question));a=tokenize(candidate)
    ids=torch.cat([q,a],-1)
    out,ranges=dense_forward(model,ids,attention,positions)
    start=ranges["query"][0]+q.shape[1]-1
    lp=out.logits[:,start:start+a.shape[1]].float().log_softmax(-1)
    vals=lp.gather(-1,a.to(lp.device).unsqueeze(-1)).squeeze(-1)[0]
    return {"candidate":candidate,"token_ids":a[0].tolist(),
        "sum_logp":vals.sum().item(),"per_token_logp":vals.tolist()}


## 4. 真实模型执行：一次只加载一个后端
对照项目包括：v5 vs 完整图；完整图重复运行；v5 重复运行；query 缓存反向排列；首 token 与完整生成结果。先做误差检查，再进行位置/连边实验。

完整图生成每一步都重新计算短输入，速度比使用缓存慢，但可避免把待验证的缓存机制再次引入参考计算。两种后端均使用 BF16，不声称等价于 FP32 精度实验。


In [8]:
for backend in BACKENDS:
    gc.collect();torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print("加载后端:",backend,flush=True)
    model=AutoModelForCausalLM.from_pretrained(str(MODEL_PATH),local_files_only=True,
        torch_dtype=torch.bfloat16,device_map={"":0},low_cpu_mem_usage=True,
        attn_implementation=backend).eval()
    assert {str(p.device) for p in model.parameters()}=={"cuda:0"}
    g=build_graphkv_from_ids(model=model,
        source_inputs={n:node_inputs[n] for n in SOURCE_IDS},
        target_inputs={n:node_inputs[n] for n in TARGET_IDS},edges=EDGES,
        prefix_input_ids=prefix_ids,chunk_span=L)
    before=cache_hash(g)
    for project,question in QUESTIONS.items():
        q=tokenize(query_text(question))
        dense,ranges=dense_forward(model,q,keep_cache=True)
        ref_logits=dense.logits[:,-1].detach().float()
        layer_diffs=[]
        for n in SOURCE_IDS+TARGET_IDS:
            a,b=ranges[n]
            for attr in ["key_cache","value_cache"]:
                for layer,(actual,full) in enumerate(zip(getattr(g.nodes[n],attr),getattr(dense.past_key_values,attr))):
                    row=difference(actual,full[:,:,a:b,:])
                    row.update({"node":n,"kind":attr,"layer":layer});layer_diffs.append(row)
        del dense
        chunk_logits=graphkv_next_logits(model=model,graph_cache=g,query_input_ids=q)
        comparison=compare_logits(chunk_logits,ref_logits)
        results["oracle"].append({"backend":backend,"project":project,
            "logits":comparison,"per_layer_KV":layer_diffs})
        repeated,_=dense_forward(model,q)
        repeat_dense=compare_logits(ref_logits,repeated.logits[:,-1]);del repeated
        repeated_chunk=graphkv_next_logits(model=model,graph_cache=g,query_input_ids=q)
        repeat_chunk=compare_logits(chunk_logits,repeated_chunk);del repeated_chunk
        reverse=graphkv_next_logits(model=model,graph_cache=g,query_input_ids=q,
            source_ids=list(reversed(SOURCE_IDS)),target_ids=list(reversed(TARGET_IDS)))
        perm=compare_logits(chunk_logits,reverse);del reverse
        results["stability"].append({"backend":backend,"project":project,
            "dense_repeat":repeat_dense,"chunk_repeat":repeat_chunk,"reverse_order":perm})
        print("对照",backend,project,"v5/full:",comparison,"repeat:",repeat_chunk["max_abs"],flush=True)
        del ref_logits,chunk_logits
        if RUN_GENERATION:
            for order,ss,tt in [("normal",SOURCE_IDS,TARGET_IDS),
                                 ("reversed",list(reversed(SOURCE_IDS)),list(reversed(TARGET_IDS)))]:
                r=greedy_generate_graphkv(model=model,tokenizer=tokenizer,graph_cache=g,
                    query=query_text(question),source_ids=ss,target_ids=tt,
                    max_new_tokens=MAX_NEW_TOKENS,return_details=True)
                results["answers"].append({"backend":backend,"condition":"v5_"+order,
                    "project":project,"answer":r.text,"token_ids":r.token_ids,
                    "eos":r.stopped_on_eos,"hit_limit":len(r.token_ids)==MAX_NEW_TOKENS and not r.stopped_on_eos,
                    "rough_pass":rough_check(project,r.text)})
        save_results()
    assert cache_hash(g)==before,"v5 推理修改了保存的节点"
    del g
    for condition,(attention,positions) in CONDITIONS.items():
        for project,question in QUESTIONS.items():
            if RUN_GENERATION:
                r=dense_generate(model,question,attention,positions)
                r.update({"backend":backend,"condition":condition,"project":project,
                    "rough_pass":rough_check(project,r["answer"])})
                results["answers"].append(r)
                print(backend,condition,project,repr(r["answer"]),flush=True)
            if project in ["ORION","LYRA"]:
                scores=[candidate_score(model,question,c,attention,positions)
                    for c in ["Copper","Silver","copper","silver"," copper"," silver"]]
                results["candidate_scores"].append({"backend":backend,"condition":condition,
                    "project":project,"query":query_text(question),"variants":scores})
            save_results()
    memory(backend+" 完成")
    results.setdefault("memory",[]).append({"backend":backend,
        "peak_allocated_GiB":torch.cuda.max_memory_allocated()/2**30})
    del model;gc.collect();torch.cuda.empty_cache();save_results()
print("全部完成:",OUTPUT_DIR)


加载后端: eager


Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.39it/s]


对照 eager ORION v5/full: {'max_abs': 0.5625, 'mean_abs': 0.048776183277368546, 'KL_a_b': 0.004084252752363682, 'prob_L1': 0.0675874724984169, 'argmax_a': 34, 'argmax_b': 34, 'top5_a': [34, 11439, 53443, 29815, 878], 'top5_b': [34, 11439, 53443, 29815, 878]} repeat: 0.0
对照 eager LYRA v5/full: {'max_abs': 0.384765625, 'mean_abs': 0.053641144186258316, 'KL_a_b': 0.004020128399133682, 'prob_L1': 0.046821098774671555, 'argmax_a': 34, 'argmax_b': 34, 'top5_a': [34, 11439, 53443, 29815, 66], 'top5_b': [34, 11439, 53443, 29815, 66]} repeat: 0.0
对照 eager NOVA v5/full: {'max_abs': 0.4140625, 'mean_abs': 0.04526097699999809, 'KL_a_b': 0.0035617388784885406, 'prob_L1': 0.06570152938365936, 'argmax_a': 11439, 'argmax_b': 11439, 'top5_a': [11439, 29815, 9173, 984, 2149], 'top5_b': [11439, 29815, 9173, 984, 2149]} repeat: 0.0
对照 eager COMBINED v5/full: {'max_abs': 0.5390625, 'mean_abs': 0.050979118794202805, 'KL_a_b': 8.386594709008932e-05, 'prob_L1': 0.002557657891884446, 'argmax_a': 11439, 'argmax_b

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.38it/s]


对照 sdpa ORION v5/full: {'max_abs': 0.375, 'mean_abs': 0.04906586557626724, 'KL_a_b': 0.0003843829035758972, 'prob_L1': 0.010207345709204674, 'argmax_a': 34, 'argmax_b': 34, 'top5_a': [34, 11439, 53443, 29815, 878], 'top5_b': [34, 11439, 53443, 29815, 878]} repeat: 0.0
对照 sdpa LYRA v5/full: {'max_abs': 0.375, 'mean_abs': 0.056462354958057404, 'KL_a_b': 0.001575724221765995, 'prob_L1': 0.049688033759593964, 'argmax_a': 34, 'argmax_b': 34, 'top5_a': [34, 11439, 53443, 29815, 66], 'top5_b': [34, 11439, 53443, 29815, 66]} repeat: 0.0
对照 sdpa NOVA v5/full: {'max_abs': 0.40625, 'mean_abs': 0.05274800583720207, 'KL_a_b': 0.001676458865404129, 'prob_L1': 0.05092546343803406, 'argmax_a': 11439, 'argmax_b': 11439, 'top5_a': [11439, 29815, 9173, 984, 2149], 'top5_b': [11439, 29815, 9173, 984, 2149]} repeat: 0.0
对照 sdpa COMBINED v5/full: {'max_abs': 0.375, 'mean_abs': 0.05535455793142319, 'KL_a_b': 0.0005431976169347763, 'prob_L1': 0.0106132160872221, 'argmax_a': 11439, 'argmax_b': 11439, 'top5_a':

## 5. 汇总与下一步如何判断
① v5 与完整图的差异若接近同顺序重复误差，且答案一致，更支持实现与完整图参考一致；BF16 下仍不能称为严格数学证明。

② 同时比较 `graph_shared` / `noedges_shared`、`graph_continuous` / `noedges_continuous`，观察有边是否有帮助；比较相同 mask 的 shared/continuous，观察位置安排的影响。

③ 若排列差异远大于重复误差，且改变答案，再比较 eager/SDPA 是否同样出现。不同后端最终文本不同，也不能单独证明代码有 bug。

④ 如果参考完整图也答错，但普通顺序答对，优先研究模型对图式注意力/共享位置的适应，而不是直接继续修改缓存代码。

⑤ 此处仍只有一个五节点样本；下一轮再扩展实体名、材料、节点顺序和多个样本。


In [9]:
import csv
columns=["backend","condition","project","answer","rough_pass","eos","hit_limit"]
with (OUTPUT_DIR/"answers.csv").open("w",encoding="utf-8-sig",newline="") as f:
    writer=csv.DictWriter(f,fieldnames=columns,extrasaction="ignore")
    writer.writeheader();writer.writerows(results["answers"])
print("backend | project | v5/full max logits diff | repeat diff | reverse diff")
for a,b in zip(results["oracle"],results["stability"]):
    print(a["backend"],a["project"],a["logits"]["max_abs"],
          b["chunk_repeat"]["max_abs"],b["reverse_order"]["max_abs"])
print("答案数:",len(results["answers"]),"候选评分组:",len(results["candidate_scores"]))
print("JSON:",OUTPUT_DIR/"graphkv_stage3_results.json")
print("CSV:",OUTPUT_DIR/"answers.csv")


backend | project | v5/full max logits diff | repeat diff | reverse diff
eager ORION 0.5625 0.0 0.28125
eager LYRA 0.384765625 0.0 0.2744140625
eager NOVA 0.4140625 0.0 0.4765625
eager COMBINED 0.5390625 0.0 0.4296875
sdpa ORION 0.375 0.0 0.3125
sdpa LYRA 0.375 0.0 0.3671875
sdpa NOVA 0.40625 0.0 0.265625
sdpa COMBINED 0.375 0.0 0.25390625
答案数: 56 候选评分组: 20
JSON: /home/lyh/hcy_work/GraphKV_refactor_v5/graphkv_stage3_outputs/graphkv_stage3_results.json
CSV: /home/lyh/hcy_work/GraphKV_refactor_v5/graphkv_stage3_outputs/answers.csv
